In [ ]:
"""
06_generate_predictions.py
----------------------------
Phase 8: trains the final XGBoost model on the FULL customer base
(not just the 80% train split used for evaluation) and generates
a predicted CLV value for every customer.

Why retrain on everything: Phase 6/7 already proved the approach
works using a held-out test set. Now that we trust it, using 100%
of the data for the final model gives the most accurate possible
predictions - this is standard practice once a model is validated.

Reads:
    data/processed/customer_features.csv

Writes:
    data/processed/predicted_clv.csv
"""

import pandas as pd
from xgboost import XGBRegressor

# ----------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------
df = pd.read_csv("data/processed/customer_features.csv")
print(f"Loaded {len(df)} customers")

# ----------------------------------------------------------------
# 2. ENCODE CATEGORICAL FEATURES (same approach as 05_modeling.py)
# ----------------------------------------------------------------
categorical_cols = ["acquisition_channel", "age_band", "region"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

feature_cols = [c for c in df_encoded.columns if c not in ["customer_id", "target_holdout_spend"]]
X = df_encoded[feature_cols]
y = df_encoded["target_holdout_spend"]

# ----------------------------------------------------------------
# 3. TRAIN FINAL MODEL ON 100% OF THE DATA
#    Same hyperparameters as the validated model in 05_modeling.py -
#    only the training data changes (all of it, not just 80%).
# ----------------------------------------------------------------
final_model = XGBRegressor(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
)
final_model.fit(X, y)

# ----------------------------------------------------------------
# 4. GENERATE PREDICTIONS FOR EVERY CUSTOMER
# ----------------------------------------------------------------
df["predicted_clv"] = final_model.predict(X)

# Predicted spend can't be negative in the real world - clip any
# small negative predictions (a known quirk of regression models
# near zero) up to 0.
df["predicted_clv"] = df["predicted_clv"].clip(lower=0)

# ----------------------------------------------------------------
# 5. SANITY CHECKS
# ----------------------------------------------------------------
print("\nPredicted CLV summary statistics:")
print(df["predicted_clv"].describe().round(2))

print("\nActual (target_holdout_spend) summary statistics, for comparison:")
print(df["target_holdout_spend"].describe().round(2))

print("\nAverage predicted CLV by acquisition channel (should mirror the actual pattern):")
comparison = df.groupby("acquisition_channel")[["target_holdout_spend", "predicted_clv"]].mean().round(2)
comparison.columns = ["actual_avg", "predicted_avg"]
print(comparison.sort_values("predicted_avg", ascending=False))

# Note: since this model was trained on all the data it's now
# predicting on, these predictions will look "too good" compared
# to the honest test-set performance reported in Phase 7. The
# R2/RMSE numbers from 05_modeling.py - not this script - are the
# numbers to quote as the model's real-world accuracy.

# ----------------------------------------------------------------
# 6. SAVE OUTPUT
# ----------------------------------------------------------------
output_cols = ["customer_id", "acquisition_channel", "age_band", "region", "predicted_clv"]
df[output_cols].to_csv("data/processed/predicted_clv.csv", index=False)
print("\nSaved: data/processed/predicted_clv.csv")